Exercise 1- Conceptual Questions
1. Q, K, V Analogy

Query (Q), Key (K), and Value (V) can be compared to a library sytem
The Query represents what a person is searching for
The Key represents the labels or categories of books in the library
The Value represents the actual information inside the books
The attention mechanism compares the query with all keys and retrieves the most relevant values

2. The Scaling Factor

The scaling factor 1/sqrt(d_k) prevents very large dot-product values
If the values become too large, the softmax output becomes extremely sharp, meaning one value becomes almost 1 and the others become almost 0
This causes very small gradients, making training unstable and slow

3. Self-Attention vs. Cross-Attention

Cross-attention allows the decoder to focus on relevant words from the encoder output while generating translations
The decoder uses its current state as Query and compares it with encoder Keys and Values
This helps generate context-aware translations

In [1]:
#Exercise 2- Implementing Scaled Dot-Product Attention
import torch
import torch.nn.functional as F

In [2]:
queries = torch.randn(1, 4, 8)
keys = torch.randn(1, 4, 8)
values = torch.randn(1, 4, 8)

print("Queries shape:", queries.shape)
print("Keys shape:", keys.shape)
print("Values shape:", values.shape)

Queries shape: torch.Size([1, 4, 8])
Keys shape: torch.Size([1, 4, 8])
Values shape: torch.Size([1, 4, 8])


In [3]:
scores = torch.matmul(queries, keys.transpose(-2, -1))

print("Scores shape:", scores.shape)
print(scores)

Scores shape: torch.Size([1, 4, 4])
tensor([[[ 2.6815, -0.2413, -1.9416,  2.3093],
         [-2.2802, -0.0405,  2.3389, -2.4397],
         [ 3.4239,  1.8383, -3.4151,  0.8718],
         [-0.9070,  2.5071,  0.3277, -2.4348]]])


In [4]:
d_k = keys.shape[-1]

scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

print(scaled_scores)

tensor([[[ 0.9480, -0.0853, -0.6865,  0.8164],
         [-0.8062, -0.0143,  0.8269, -0.8626],
         [ 1.2105,  0.6499, -1.2074,  0.3082],
         [-0.3207,  0.8864,  0.1159, -0.8608]]])


In [5]:
attention_weights = F.softmax(scaled_scores, dim=-1)

print("Attention weights:")
print(attention_weights)

Attention weights:
tensor([[[0.4119, 0.1466, 0.0803, 0.3611],
         [0.1078, 0.2381, 0.5521, 0.1019],
         [0.4841, 0.2764, 0.0431, 0.1964],
         [0.1545, 0.5165, 0.2390, 0.0900]]])


In [6]:
def scaled_dot_product_attention(q, k, v):
    
    scores = torch.matmul(q, k.transpose(-2, -1))
    
    d_k = k.shape[-1]
    
    scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    attention_weights = F.softmax(scaled_scores, dim=-1)
    
    output = torch.matmul(attention_weights, v)
    
    return output, attention_weights

In [7]:
#Exercise 3- Challenge Problem - Masking
def masked_scaled_dot_product_attention(q, k, v, mask=None):

    scores = torch.matmul(q, k.transpose(-2, -1))

    d_k = k.shape[-1]

    scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask, -1e9)

    attention_weights = F.softmax(scaled_scores, dim=-1)

    output = torch.matmul(attention_weights, v)

    return output, attention_weights

In [8]:
mask = torch.triu(torch.ones(4, 4), diagonal=1).bool()

print(mask)

tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])


In [9]:
masked_output, masked_attention = masked_scaled_dot_product_attention(
    queries,
    keys,
    values,
    mask
)

print("Masked attention weights:")
print(masked_attention)

print("\nMasked output:")
print(masked_output)

Masked attention weights:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.3118, 0.6882, 0.0000, 0.0000],
         [0.6024, 0.3439, 0.0537, 0.0000],
         [0.1545, 0.5165, 0.2390, 0.0900]]])

Masked output:
tensor([[[-1.2345,  0.1048,  0.6341, -0.1965, -0.3185,  1.1343,  1.2022,
          -2.1850],
         [-1.5282, -1.4677,  0.0986, -0.0947, -0.2117, -0.0630,  0.2333,
          -0.3679],
         [-1.2897, -0.6954,  0.3099, -0.1165, -0.2563,  0.4441,  0.6441,
          -1.2017],
         [-1.0246, -1.1289, -0.0242, -0.1628, -0.0861, -0.4139,  0.0633,
          -0.3253]]])


Why masking works

The masked positions are replaced with a very large negative number (-1e9)
When softmax is applied, these values become almost zero probability
This prevents the model from attending to future tokens during decoding